# Naive Bayes Classification - NBA Player Longevity Prediction

**Author:** Lapele Kenneth  
**Program:** 3MTT AI and Machine Learning Fellowship  
**Dataset:** NBA Engineered Features (1,340 players, 10 features)

---

## Objective

Build a Gaussian Naive Bayes classifier to predict whether an NBA player will last **5 or more years** in the league (`target_5yrs = 1`).  
We evaluate performance using Confusion Matrix, Precision, and Recall aligned with real scouting business priorities.

## Step 1 - Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score,
    f1_score, accuracy_score
)

pd.set_option('display.max_columns', None)
print('Libraries loaded successfully')

## Step 2 - Load Dataset and Identify Target Variable

The dataset contains performance statistics for 1,340 NBA players.  
The target variable is `target_5yrs`:
- **1** = player lasted 5 or more years in the NBA
- **0** = player did not reach 5 years

All other columns serve as predictors (independent variables).

In [ ]:
df = pd.read_csv('nba_engineered.csv')

print('Dataset shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Null values:', df.isnull().sum().sum())
print('Target distribution:')
print(df['target_5yrs'].value_counts())
print(f'Class balance: {df["target_5yrs"].mean()*100:.1f}% lasted 5+ years')
df.head()

## Step 3 - Preprocessing: Feature and Target Separation

We separate the dataset into:
- **X** = all feature columns (predictors)
- **y** = target column (`target_5yrs`)

We then apply **StandardScaler** to normalise all features to mean=0 and std=1.  
This ensures features with large ranges (like `total_points`) do not dominate  
the Gaussian probability calculations over features with small ranges (like `fg%`).

In [ ]:
X = df.drop(columns=['target_5yrs'])
y = df['target_5yrs']

print('Features (X) shape:', X.shape)
print('Target (y) shape:', y.shape)
X.describe().round(3)

## Step 4 - Train and Test Split

We split the data **80% training / 20% testing**.  
We use `stratify=y` to ensure both splits maintain the same class ratio  
as the original dataset, preventing imbalanced evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples    : {X_test.shape[0]}')
print(f'Train class balance: {y_train.mean()*100:.1f}% positive')
print(f'Test class balance : {y_test.mean()*100:.1f}% positive')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, split, label in zip(axes, [y_train, y_test], ['Training Set (80%)', 'Test Set (20%)']):
    counts = split.value_counts()
    ax.bar(['Less than 5 Years', '5 or More Years'], [counts[0], counts[1]],
           color=['#e74c3c', '#2ecc71'], edgecolor='white', width=0.5)
    ax.set_title(label, fontsize=12)
    ax.set_ylabel('Player Count')
    for i, v in enumerate([counts[0], counts[1]]):
        ax.text(i, v + 2, str(v), ha='center', fontweight='bold')
plt.suptitle('Class Distribution: Train vs Test Split (Stratified)', fontsize=13)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Scaling complete. Training feature means (should be near 0):')
print(X_train_scaled.mean(axis=0).round(3))

## Step 5 - Why Gaussian Naive Bayes? The Naive Assumption Explained

### Why GaussianNB Was Chosen

Gaussian Naive Bayes is well suited for this dataset because:
- All features are **continuous numeric variables** (shooting percentages, rebounds, assists, etc.)
- GaussianNB models each feature as following a **Gaussian (normal) distribution** within each class
- It is fast to train, easy to interpret, and works well as a baseline classification model
- It produces **probability scores** that can be used to rank and prioritise scouting targets

### The Naive Independence Assumption

Naive Bayes is called **naive** because it assumes all features are **conditionally independent**  
given the class label. This means the model treats each statistic as if it has  
no relationship with any other statistic when predicting career length.

Mathematically, the model computes:

```
P(5yrs+ | stats) is proportional to P(5yrs+) x P(fg|5yrs+) x P(reb|5yrs+) x P(ast|5yrs+) x ...
```

### Is This Assumption Realistic for Basketball?

**No - and this is the key limitation of this model.**  
Basketball statistics are naturally correlated because they all reflect playing time and role:

| Feature Pair | Why They Are Correlated |
|---|---|
| `total_points` and `tov` | Players who score more also handle the ball more, creating more turnover chances |
| `total_points` and `reb` | High scorers are often active on the boards |
| `ast` and `tov` | Playmakers who create assists also risk more turnovers |
| `fg` and `efficiency` | Field goal percentage is a direct input to the efficiency metric |

Despite this violation, Gaussian Naive Bayes still produces useful predictions  
because the pattern differences between classes are strong enough to overcome the assumption error.  
This is known as the **Naive Bayes paradox** - it works in practice even when the assumption is wrong.

In [ ]:
corr = X.corr()
plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 9})
plt.title('Feature Correlations - Violations of the Independence Assumption', fontsize=12)
plt.tight_layout()
plt.savefig('independence_violation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Pairs with |r| > 0.5 (strongest independence violations):')
for col in corr.columns:
    for row in corr.index:
        if col != row and abs(corr.loc[row, col]) > 0.5 and row < col:
            print(f'  {row} vs {col}: r = {corr.loc[row, col]:.3f}')

## Step 6 - Train the Gaussian Naive Bayes Model

In [ ]:
model = GaussianNB()
model.fit(X_train_scaled, y_train)

print('Model trained successfully.')
print(f'Class prior probabilities:')
print(f'  P(less than 5 years) = {model.class_prior_[0]:.3f}')
print(f'  P(5 or more years)   = {model.class_prior_[1]:.3f}')

## Step 7 - Confusion Matrix

The confusion matrix breaks down predictions into four categories:

| | Predicted: Less than 5 Years | Predicted: 5 or More Years |
|---|---|---|
| **Actual: Less than 5 Years** | True Negative (TN) - correct rejection | False Positive (FP) - bust risk |
| **Actual: 5 or More Years** | False Negative (FN) - missed talent | True Positive (TP) - correct pick |

In [ ]:
y_pred = model.predict(X_test_scaled)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred: Less than 5yrs', 'Pred: 5 or More yrs'],
            yticklabels=['True: Less than 5yrs', 'True: 5 or More yrs'],
            annot_kws={'size': 16})
ax.set_title('Confusion Matrix - Gaussian Naive Bayes', fontsize=13, pad=15)
ax.set_ylabel('Actual Label', fontsize=11)
ax.set_xlabel('Predicted Label', fontsize=11)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'True Negatives  (TN): {tn} - correctly identified short-career players')
print(f'False Positives (FP): {fp} - predicted long career, actually short (BUST RISK)')
print(f'False Negatives (FN): {fn} - predicted short career, actually long (MISSED TALENT)')
print(f'True Positives  (TP): {tp} - correctly identified long-career players')

## Step 8 - Precision, Recall and Business Interpretation

### Metric Definitions

| Metric | Formula | What It Measures |
|---|---|---|
| Precision | TP divided by (TP + FP) | Of all players predicted to last 5+ years, how many actually did? |
| Recall | TP divided by (TP + FN) | Of all players who actually lasted 5+ years, how many did we identify? |
| F1 Score | Harmonic mean of Precision and Recall | Balanced metric when both matter |
| Accuracy | Correct predictions divided by Total | Overall correctness |

### Business Interpretation for NBA Scouting

**Precision = 80%**  
When our model recommends a player as a long-career prospect, it is correct 8 out of 10 times.  
The remaining 2 out of 10 are **False Positives** - players who were signed but did not last.  
In NBA terms, a False Positive is a **bust** - a player given a contract who fails to stick in the league.  
This wastes roster spots, salary cap space, and coaching resources.  
Our 80% precision means the model is a reliable tool for **shortlisting contract candidates**.

**Recall = 55%**  
The model correctly identifies only 55% of players who actually lasted 5+ years.  
The remaining 45% are **False Negatives** - talented players the model incorrectly passed over.  
In NBA terms, a False Negative is a **missed star** - a future long-career player lost to a competitor.  
This is a significant weakness. Missing nearly half of genuinely talented players is costly in a competitive draft.

### The Precision-Recall Trade-off

This model **prioritises precision over recall**.  
It is conservative - it only flags a player as a long-career prospect when it is quite confident.  
This makes it more useful for **final contract decisions** (where busts are expensive)  
but less suitable as a **broad talent discovery tool** (where missing stars is costly).

**Recommendation:** Use this model to filter and rank candidates, but pair it  
with human scouting review to recover the missed talent (False Negatives).

In [ ]:
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print('MODEL PERFORMANCE SUMMARY')
print('='*40)
print(f'Accuracy  : {acc:.4f} ({acc*100:.1f}%)')
print(f'Precision : {prec:.4f} ({prec*100:.1f}%)')
print(f'Recall    : {rec:.4f} ({rec*100:.1f}%)')
print(f'F1 Score  : {f1:.4f} ({f1*100:.1f}%)')
print()
print(classification_report(y_test, y_pred, target_names=['Less than 5 Years', '5 or More Years']))

In [ ]:
metrics = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1 Score': f1}
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
bars = ax.bar(metrics.keys(), metrics.values(), color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_title('Model Performance Metrics - Gaussian Naive Bayes', fontsize=13)
ax.set_ylabel('Score')
ax.axhline(0.7, color='gray', linestyle='--', alpha=0.6, label='0.70 reference line')
ax.legend()
plt.tight_layout()
plt.savefig('model_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9 - Feature Importance Analysis

In Gaussian Naive Bayes, we estimate feature importance by calculating  
the **absolute difference between class means** for each feature after scaling.  
A larger difference means the feature better separates long-career from short-career players.

In [ ]:
means_class0 = model.theta_[0]
means_class1 = model.theta_[1]
importance = np.abs(means_class1 - means_class0)

feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Mean_LessThan5': means_class0.round(3),
    'Mean_5Plus': means_class1.round(3),
    'Importance': importance.round(3)
}).sort_values('Importance', ascending=False)

print('Feature Importance (Class Mean Difference):')
print(feat_imp.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 6))
sorted_imp = feat_imp.sort_values('Importance')
colors = ['#2ecc71' if v > feat_imp['Importance'].median() else '#3498db'
          for v in sorted_imp['Importance']]
ax.barh(sorted_imp['Feature'], sorted_imp['Importance'], color=colors, edgecolor='white')
ax.axvline(feat_imp['Importance'].median(), color='red', linestyle='--',
           alpha=0.7, label='Median importance')
ax.set_title('Feature Importance - Class Mean Difference', fontsize=13)
ax.set_xlabel('Absolute difference between class means (scaled units)')
ax.legend()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10 - Stakeholder Summary: Scouting Department Report

### What the Model Found

After evaluating 268 players the model had never seen before, here are the results in plain language:

| Outcome | Count | Business Meaning |
|---|---|---|
| Correctly recommended long-career players | 92 | Safe contract targets |
| Players wrongly recommended (False Positives) | 23 | Potential busts - wasted investment |
| Long-career players we missed (False Negatives) | 74 | Talent lost to competitors |
| Correctly filtered out short-career players | 79 | Avoided poor investments |

### When to Trust This Model

**Use the model when:** You need to quickly shortlist candidates from a large pool.  
The 80% precision means most players it recommends are genuine long-career prospects.

**Question the model when:** You are doing broad talent searches.  
The 55% recall means it misses nearly half of all actual long-career players.

### Key Risk: The False Positive Problem

A False Positive in NBA scouting means offering a contract to a player  
who will not last 5 years in the league. The costs include:
- Wasted salary cap space that could have signed a better player
- Lost roster spot blocking development of younger talent
- Coaching and training resources spent on a non-viable player

Our model produces only 23 False Positives out of 115 positive predictions (20% bust rate),  
which compares favourably to pure guesswork or unstructured scouting.

### Recommended Workflow

1. Run the model to rank all draft prospects by predicted longevity probability
2. Automatically shortlist the top 60% by model confidence
3. Apply human expert review to borderline cases, especially players with high `efficiency_rating` but low model score
4. For final contract decisions, upgrade to Random Forest or XGBoost which capture feature interactions the Naive Bayes model ignores

### Top 5 Features Driving Longevity Predictions

1. `total_points` - strongest separator between career lengths
2. `reb` - reflects playing time and team value
3. `tov` - high for long-career players because they handle the ball more
4. `stl` - defensive engagement and court time
5. `fg` - shooting efficiency is a consistent longevity signal

In [ ]:
print('FINAL MODEL SUMMARY')
print('='*50)
print(f'Model          : Gaussian Naive Bayes')
print(f'Dataset        : {df.shape[0]} players, {X.shape[1]} features')
print(f'Train/Test     : 80% / 20% stratified split')
print(f'Accuracy       : {acc*100:.1f}%')
print(f'Precision      : {prec*100:.1f}% - 1 in 5 recommendations is a bust')
print(f'Recall         : {rec*100:.1f}% - catches just over half of all long-career players')
print(f'F1 Score       : {f1*100:.1f}%')
print()
print('Top Features   : total_points, reb, tov, stl, fg')
print('Limitation     : Independence assumption violated by correlated stats')
print('Recommendation : Use as first-pass filter; validate with human scouting and ensemble models')